## 이벤트별 집계 확인

이 코드는 사용자의 서비스 전환 과정을 파악하기 위해, 퍼널(Funnel) 분석 대상이 될 핵심 이벤트를 선별하고 집계하는 단계임. **AARRR 프레임워크의 Activation(활성화) 단계**에서 사용자가 핵심 가치를 경험하는 지점을 파악하는 데 활용됨.

* **`event_name` 기준 집계:** 서비스 내 발생하는 전체 이벤트 종류 및 발생 건수 확인
* **사용자 수(`unique_users`) 함께 확인:** 단순히 총 발생 수만 보는 것이 아니라, 실제 해당 행동을 경험한 사용자 기반의 규모 측정

### 주요 확인 포인트
* **이벤트 라인업 파악:** Activation 퍼널을 구성할 핵심 행동(이벤트)이 정상적으로 수집되고 있는지 확인
* **Activation 퍼널 이벤트 선별:** 가입 이후 첫 핵심 기능 이용, 주요 페이지 방문 등 퍼널 단계별 기준이 될 이벤트를 결정


In [1]:
from google.cloud import bigquery

client = bigquery.Client(project="pro-talon-503713-s3")

query = """
SELECT
    event_name,
    COUNT(*) AS event_count,
    COUNT(DISTINCT user_pseudo_id) AS unique_users
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
GROUP BY event_name
ORDER BY event_count DESC
"""

df = client.query(query).to_dataframe()
print(df)

             event_name  event_count  unique_users
0             page_view      1350428        269792
1       user_engagement      1058721        213004
2                scroll       493072        138098
3             view_item       386068         61252
4         session_start       354970        267116
5           first_visit       257462        257314
6        view_promotion       190104        102443
7           add_to_cart        58543         12545
8        begin_checkout        38757          9715
9           select_item        31007         13180
10  view_search_results        26172         14449
11    add_shipping_info        19722          9714
12     add_payment_info        13899          5751
13     select_promotion         9450          8164
14             purchase         5692          4419
15                click         1446          1010
16       view_item_list           71            44


# GA4 주요 이벤트 정의
Google Analytics 4(GA4)에서 수집되는 주요 이벤트를 공식 이벤트 정의 기준으로 정리한 문서입니다.

| 구분 | 이벤트 | 정의 및 발생 조건 | 분석 시 주요 의미 |
|:---:|:---|:---|:---|
| 자동 수집 | `page_view` | 페이지가 로드되거나, 활성 사이트에서 브라우저 기록 상태가 변경될 때 발생합니다. | 페이지 조회 및 화면 전환량 측정 |
| 자동 수집 | `scroll` | 사용자가 각 페이지에서 처음으로 하단에 도달했을 때 발생합니다. 웹에서는 세로 기준으로 페이지의 90% 이상이 표시된 경우를 의미합니다. | 콘텐츠 도달 및 페이지 참여도 측정 |
| 자동 수집 | `session_start` | 사용자가 앱 또는 웹사이트에 참여할 때 발생합니다. 세션 ID와 세션 번호는 세션마다 자동으로 생성되며 세션의 각 이벤트에 연결됩니다. | 세션 수와 방문 시작 시점 파악 |
| 자동 수집 | `user_engagement` | 앱이 포그라운드에 있거나 웹페이지가 최소 1초간 포커스 상태일 때 발생합니다. | 실제 사용자 참여 시간 및 활성도 측정 |
| 자동 수집 | `view_search_results` | URL에 검색어 쿼리 매개변수가 포함되어 사용자가 사이트 검색을 수행한 것으로 간주될 때마다 발생합니다. | 사이트 검색 이용 현황 및 검색어 분석 |
| 자동 수집 | `first_visit` | 사용자가 웹사이트를 처음 방문하거나, 애널리틱스를 사용하는 Android 인스턴트 앱을 처음 실행할 때 발생합니다. | 신규 사용자 유입 및 최초 방문 분석 |
| 전자상거래 | `add_payment_info` | 사용자가 결제 과정에서 결제 정보를 제출할 때 발생합니다. | 결제 정보 입력 단계의 진행 및 이탈 분석 |
| 전자상거래 | `add_shipping_info` | 사용자가 결제 과정에서 배송 정보를 제출할 때 발생합니다. | 배송 정보 입력 단계의 진행 및 이탈 분석 |
| 전자상거래 | `add_to_cart` | 사용자가 상품을 장바구니에 추가할 때 발생합니다. | 상품 관심도 및 장바구니 전환 분석 |
| 전자상거래 | `begin_checkout` | 사용자가 결제 프로세스를 시작할 때 발생합니다. | 장바구니에서 결제로 이동한 사용자 분석 |
| 전자상거래 | `purchase` | 사용자가 구매를 완료할 때 발생합니다. | 매출, 구매 건수 및 구매 전환 분석 |
| 전자상거래 | `view_item` | 사용자가 상품을 조회할 때 발생합니다. | 개별 상품 조회 및 상품 관심도 분석 |
| 전자상거래 | `view_item_list` | 사용자가 상품 또는 서비스 목록을 조회할 때 발생합니다. | 목록 노출 및 상품 탐색 분석 |
| 프로모션 | `view_promotion` | 사용자가 웹사이트 또는 앱에서 프로모션을 조회할 때 발생합니다. | 프로모션 노출 및 도달 분석 |
| 프로모션 | `select_promotion` | 사용자가 프로모션을 선택할 때 발생합니다. | 프로모션 클릭 및 프로모션 유입 분석 |
| 상품 목록 상호작용 | `select_item` | 사용자가 상품 또는 서비스 목록에서 항목을 선택할 때 발생합니다. | 목록에서 개별 상품으로 이동한 행동 분석 |
| 사용자 상호작용 | `click` | 사용자가 현재 도메인에서 나가는 링크를 클릭할 때마다 발생합니다. | 외부 링크 클릭 및 이탈 경로 분석 |

# Activation 퍼널 분석 계획

> **목표:** 상품을 조회한 사용자가 구매까지 이어지는 과정을 단계별로 추적하고, 각 단계의 이탈 지점을 파악한다.

## 1. 핵심 질문

**상품을 조회한 사용자 중 몇 명이 구매까지 도달하며, 가장 큰 이탈은 어느 단계에서 발생하는가?**

본 분석의 최종 목적지는 구매 완료이며, 퍼널 각 단계의 도달률과 이탈 규모를 산출한다. 방문 자체보다 상품에 관심을 보인 시점부터 여정을 추적하기 위해 `view_item`을 퍼널 시작점으로 둔다.

활성화 기준 이벤트(어느 단계 도달을 '활성화'로 정의할 것인가)는 본 퍼널 결과와 후속 리텐션·LTV 검증을 통해 확정한다.

## 2. 분석 퍼널

| 순서 | 사용자 행동 | 이벤트 |
|:---:|:---|:---|
| 1 | 상품 조회 | `view_item` |
| 2 | 장바구니 담기 | `add_to_cart` |
| 3 | 결제 시작 | `begin_checkout` |
| 4 | 배송정보 입력 | `add_shipping_info` |
| 5 | 결제정보 입력 | `add_payment_info` |
| 6 | 구매 완료 | `purchase` |

### 시작점을 `view_item`으로 정한 이유

- 본 분석은 방문 이후 상품에 관심을 보인 사용자가 구매까지 이어지는지에 초점을 둔다.
- `view_item_list`는 분석 기간 내 발생 건수가 71건에 불과해 퍼널 단계로서 유의미하지 않아 제외한다.
- `select_item`은 광고나 공유 링크를 통한 유입 시 목록 조회 없이 `view_item`으로 직행할 수 있어 필수 경로가 아니므로 제외한다. (※ 실제 발생 건수 확인 후 수치 근거 보완)

## 3. 분석 설계

### 3-1. 분석 기간

**2020-11-02(월) ~ 2021-01-31(일), 91일 = 13주**

데이터셋 보유 기간은 2020-11-01부터이나, 사용자 유치 분석의 주 단위 집계와 모집단을 일치시키기 위해 일요일인 2020-11-01을 제외한다. 그 결과 분석 기간은 월요일 시작, 일요일 종료의 완전한 13주로 구성된다.

### 3-2. 집계 단위와 모집단

| 구분 | 정의 |
|:---|:---|
| 집계 단위 | `user_pseudo_id` (고유 사용자 1명 = 1) |
| 모집단 | 분석 기간 내 `first_visit` 이벤트 보유자 (신규 사용자) |
| 퍼널 1단계 | 모집단 중 `view_item` 도달자 |

`user_id`는 해당 데이터셋에 수집되어 있지 않아(전 구간 0건) 로그인 기반 개인 식별이 불가능하다. 모집단을 신규 사용자로 한정하는 것은 기존 재방문 고객이 섞여 전환율이 과대 추정되는 것을 방지하기 위함이다.

### 3-3. 단계 도달 판정 규칙

각 단계는 **직전에 도달한 단계 이후에 발생해야** 도달로 인정한다(시간순 검증). 단, 선행 단계를 모두 완료할 것은 요구하지 않으며 단계 건너뛰기를 허용한다.

- 기준 시점은 "직전 순번 단계"가 아니라 "실제로 도달한 마지막 단계"로 하며, 도달한 이전 단계가 없으면 퍼널 진입 시점을 기준으로 한다.
- GA4는 이벤트를 배치로 전송하므로 서로 다른 이벤트가 동일한 `event_timestamp`를 가질 수 있다. 따라서 선후 판정은 역행만 배제하는 방식(`>=`)을 적용한다.
- 동일 사용자가 같은 이벤트를 여러 번 발생시켜도 1명으로 집계한다.
- 세션을 구분하지 않고 사용자 단위로 집계한다.

### 3-4. 전환율 산출

| 지표 | 계산식 |
|:---|:---|
| 퍼널 진입률 | `view_item` 도달자 ÷ 신규 사용자 |
| 순차 전환율 | 해당 단계 도달자 ÷ 직전 단계 도달자 |
| 누적 전환율 | 해당 단계 도달자 ÷ `view_item` 도달자 |

순차 전환율로 병목 구간을, 누적 전환율로 전체 손실 규모를 파악한다.

## 4. 분석의 한계

1. **기기 단위 식별** — `user_pseudo_id`는 쿠키·브라우저 기반 식별자다. 다기기 사용자는 복수 사용자로 분리 집계되므로, 순 사용자 수는 **과대**, 전환율은 **과소** 추정되는 방향으로 편향된다.

2. **선행 단계 완료 미요구** — 시간 순서는 검증하지만 선행 단계를 모두 완료했을 것은 요구하지 않는다. 따라서 특정 단계 도달자 중에는 이전 단계 이벤트가 기록되지 않은 사용자가 포함될 수 있으며, 하위 단계 인원이 상위 단계를 초과하는 역전 현상이 발생할 수 있다. 이는 계측 누락과 정상적인 단계 생략(저장된 결제수단 사용 등)을 배제하지 않기 위한 설계상 선택이며, 해당 사례는 별도 품질 지표로 보고한다.

3. **상품 단위 미매칭** — 조회한 상품과 구매한 상품의 동일성을 확인하지 않는다. 따라서 본 지표는 상품별 전환이 아니라 사용자의 단계 도달 여부를 측정한다.

4. **인과관계 미보장** — 세션을 구분하지 않고 사용자 단위로 집계하므로 수일 간격으로 떨어진 이벤트도 하나의 지표에 함께 반영된다. 그러나 이들이 연속된 의사결정의 결과라는 근거는 없으며, 그 사이에 개입한 외부 요인(광고 노출, 외부 추천 등)은 관측되지 않는다. 짧은 윈도우를 설정할수록 이 비약은 줄어든다.

5. **관측 기간 불균등** — 현 단계에서는 관측 윈도우를 적용하지 않으므로 초기 유입자가 후기 유입자보다 긴 관찰 기간을 갖는다. 결과적으로 전환율은 초기 코호트에 유리하게 산출된다. 전체 기간 결과 확인 후 첫 방문→구매 소요 시간 분포를 근거로 윈도우를 설정하여 해소한다.